# OGB Graph Property Prediction with GNNVisualizer

This notebook builds graph-level GCN, GAT, GraphSAGE, and GIN models for an Open Graph Benchmark graph property prediction dataset, then renders the four trained models with `GNNVisualizer`.

The default dataset is `ogbg-molhiv`, the small OGB molecular benchmark. The OGB graph-property docs list `ogbg-molhiv` as a binary graph classification task with 9-dimensional atom features, and expose the PyG loader through `PygGraphPropPredDataset`.

Source docs: [OGB graph property prediction](https://ogb.stanford.edu/docs/graphprop/).

If imports fail in a fresh kernel, install the runtime packages first:

```bash
python3 -m pip install torch torch-geometric ogb
```

Set `OGB_GRAPH_DATASET` before running the notebook to point at another compatible OGB graph-property dataset. This demo keeps the raw encoded node features as floats so the first visualized convolution consumes exactly the displayed node feature matrix.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, GCNConv, GINConv, SAGEConv, global_mean_pool

from gnn_exp import GNNVisualizer

from ogb.graphproppred import PygGraphPropPredDataset


In [ ]:
SEED = 7
torch.manual_seed(SEED)

DATASET_NAME = os.environ.get("OGB_GRAPH_DATASET", "ogbg-molhiv")
DATA_ROOT = repo_root / "data" / "ogb"
MAX_TRAIN_GRAPHS = int(os.environ.get("OGB_MAX_TRAIN_GRAPHS", "256"))
BATCH_SIZE = 32
EPOCHS = 4
HIDDEN_CHANNELS = 16


def ensure_node_features(data):
    data = data.clone()
    if getattr(data, "x", None) is None:
        row = data.edge_index[0]
        degree = torch.bincount(row, minlength=data.num_nodes).float().view(-1, 1)
        scale = degree.max().clamp_min(1.0)
        data.x = degree / scale
    else:
        data.x = data.x.float()
    return data


dataset = PygGraphPropPredDataset(name=DATASET_NAME, root=str(DATA_ROOT))
split_idx = dataset.get_idx_split()
train_indices = split_idx["train"][: min(MAX_TRAIN_GRAPHS, len(split_idx["train"]))].tolist()
train_graphs = [ensure_node_features(dataset[int(index)]) for index in train_indices]
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
visual_data = train_graphs[0]
QUERY_PAIR = [0, min(1, visual_data.num_nodes - 1)]

sample_y = visual_data.y.view(1, -1)
OUT_CHANNELS = sample_y.size(1)
TASK_TYPE = str(getattr(dataset, "task_type", "binary classification")).lower()
NUM_FEATURES = visual_data.num_features

display(Markdown(
    f"Dataset `{DATASET_NAME}` loaded with **{len(dataset):,} graphs**. "
    f"This notebook trains on **{len(train_graphs):,} graphs** for a fast visual demo. "
    f"The visualized graph has **{visual_data.num_nodes} nodes**, "
    f"**{visual_data.edge_index.size(1)} directed edges**, "
    f"**{NUM_FEATURES} node features**, and **{OUT_CHANNELS} prediction target(s)**."
))


In [ ]:
class GCNGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GATGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        if hidden_channels % 2 != 0:
            raise ValueError("hidden_channels must be divisible by 2")
        heads = 2
        per_head_channels = hidden_channels // heads
        self.conv1 = GATConv(in_channels, per_head_channels, heads=heads, concat=True)
        self.act1 = nn.Tanh()
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GraphSAGEGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.act1 = nn.Tanh()
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


class GINGraphModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act1 = nn.Tanh()
        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, hidden_channels),
        ))
        self.act2 = nn.Tanh()
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch=None):
        x = x.float()
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        h = self.act1(self.conv1(x, edge_index))
        h = self.act2(self.conv2(h, edge_index))
        graph_embedding = global_mean_pool(h, batch)
        return self.classifier(graph_embedding)


In [ ]:
def target_from_batch(batch):
    target = batch.y.float()
    if target.dim() == 1:
        target = target.view(-1, 1)
    return target


def masked_loss(logits, target):
    mask = ~torch.isnan(target)
    if "regression" in TASK_TYPE:
        return F.mse_loss(logits[mask], target[mask])
    return F.binary_cross_entropy_with_logits(logits[mask], target[mask])


def evaluate_model(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index, batch.batch)
            target = target_from_batch(batch)
            losses.append(float(masked_loss(logits, target)))
    return sum(losses) / max(len(losses), 1)


def train_model(model, loader, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    losses = []
    for _ in range(epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = masked_loss(logits, target_from_batch(batch))
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
    return {"final_loss": losses[-1], "mean_loss": evaluate_model(model, loader)}


model_builders = {
    "GCN": lambda: GCNGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GAT": lambda: GATGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GraphSAGE": lambda: GraphSAGEGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
    "GIN": lambda: GINGraphModel(NUM_FEATURES, HIDDEN_CHANNELS, OUT_CHANNELS),
}

models = {}
metrics_by_model = {}
for name, build_model in model_builders.items():
    torch.manual_seed(SEED)
    model = build_model()
    metrics_by_model[name] = train_model(model, train_loader)
    models[name] = model.eval()

rows = ["| Model | Final loss | Train mean loss |", "|---|---:|---:|"]
for name, metrics in metrics_by_model.items():
    rows.append(f"| {name} | {metrics['final_loss']:.4f} | {metrics['mean_loss']:.4f} |")
display(Markdown("\n".join(rows)))


The next cell creates one `GNNVisualizer` per trained model. `mode='graph'` records the message-passing layers, classifier output, and captured global mean pooling step.

In [ ]:
EXPECTED_LAYER_TYPES = {
    "GCN": "GCNConv",
    "GAT": "GATConv",
    "GraphSAGE": "SAGEConv",
    "GIN": "GINConv",
}


def make_visualizer(model, graph_data, query_pair):
    visualizer = GNNVisualizer(renderer="svg")
    visualizer.add_model(
        data=graph_data,
        model=model.eval(),
        subgraphSample=False,
        queries=[query_pair],
        mode="graph",
    )
    return visualizer


visualizers = {
    name: make_visualizer(model, visual_data, QUERY_PAIR)
    for name, model in models.items()
}

summary_rows = [
    "| Model | First layer | Aggregation | Graph pooling | Hidden width | Visualized nodes | Query |",
    "|---|---:|---:|---:|---:|---:|---:|",
]

for name, visualizer in visualizers.items():
    first_layer = visualizer.modelInfo["conv1"]
    assert first_layer["type"] == EXPECTED_LAYER_TYPES[name]
    assert len(visualizer.graphData["x"]) == visual_data.num_nodes
    assert "graphAggregation" in visualizer.intmData
    assert len(visualizer.intmData["act1"][0]) == HIDDEN_CHANNELS
    summary_rows.append(
        f"| {name} | `{first_layer['type']}` | `{first_layer.get('aggregation')}` | "
        f"`{visualizer.intmData['graphAggregation']['type']}` | "
        f"{len(visualizer.intmData['act1'][0])} | {len(visualizer.graphData['x'])} | "
        f"`{visualizer.queries}` |"
    )

display(Markdown("\n".join(summary_rows)))


## GCN

In [ ]:
display(visualizers["GCN"])

## GAT

In [ ]:
display(visualizers["GAT"])

## GraphSAGE

In [ ]:
display(visualizers["GraphSAGE"])

## GIN

In [ ]:
display(visualizers["GIN"])